# Notebook 05: EDM Model Evaluation — ΠGDM vs FA-KGD+FPDC

**CS231N Final Project — FA-KGD: Frequency-Adaptive Kalman-Guided Diffusion**

This notebook evaluates FA-KGD with a **real pre-trained EDM score network** (65M params,
from [ADPS](https://github.com/utcsilab/ambient-diffusion-mri)) on fastMRI data.

**Contents:**
1. Load and compare saved reconstruction results
2. Visual comparison: ground truth vs ΠGDM vs FA-KGD
3. Error maps and frequency-domain analysis
4. Ablation: M-step modes and FPDC schedule direction
5. Summary results table

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from src.samplers.mri_forward import fft2c, build_radius_grid

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

OUTPUTS_DIR = Path('../outputs')
print('Available result directories:')
for d in sorted(OUTPUTS_DIR.iterdir()):
    if d.is_dir() and (d / 'results.json').exists():
        with open(d / 'results.json') as f:
            s = json.load(f).get('summary', {})
        print(f'  {d.name:25s}  N={s.get("num_slices","?"):>2}  '
              f'ΠGDM={s.get("pigdm_mean_psnr",0):.2f}  '
              f'FA-KGD={s.get("fakgd_mean_psnr",0):.2f}  '
              f'Δ={s.get("delta_psnr_mean",0):+.2f} dB')

## 1. Load EDM reconstruction results

In [ ]:
# Load the main R=4 and R=8 EDM results
def load_results(run_name):
    run_dir = OUTPUTS_DIR / run_name
    with open(run_dir / 'results.json') as f:
        meta = json.load(f)
    
    slices = []
    for vol_dir in sorted(run_dir.iterdir()):
        if not vol_dir.is_dir():
            continue
        for pt_file in sorted(vol_dir.glob('*.pt')):
            data = torch.load(pt_file, map_location='cpu', weights_only=False)
            slices.append(data)
    return meta, slices

# EDM results
meta_r4, slices_r4 = load_results('edm_5slices')
meta_r8, slices_r8 = load_results('edm_5slices_R8')

print(f'R=4: {len(slices_r4)} slices loaded')
print(f'R=8: {len(slices_r8)} slices loaded')

## 2. Visual comparison: ground truth vs ΠGDM vs FA-KGD+FPDC

In [ ]:
def compute_psnr(gt, recon):
    """Compute PSNR on magnitude images."""
    gt_mag = gt.abs()
    recon_mag = recon.abs()
    mse = ((gt_mag - recon_mag) ** 2).mean()
    if mse < 1e-12:
        return 100.0
    return (10 * torch.log10(gt_mag.max()**2 / mse)).item()

def plot_comparison(slices_data, n_show=3, title_prefix=''):
    """Plot ground truth, ΠGDM, FA-KGD, and error maps side by side."""
    n = min(n_show, len(slices_data))
    fig, axes = plt.subplots(n, 5, figsize=(20, 4*n))
    if n == 1:
        axes = axes[np.newaxis, :]
    
    for i in range(n):
        d = slices_data[i]
        gt = d['x_gt'].abs()
        pigdm = d['pigdm_recon'].abs()
        fakgd = d['fakgd_recon'].abs()
        
        psnr_p = compute_psnr(d['x_gt'], d['pigdm_recon'])
        psnr_f = compute_psnr(d['x_gt'], d['fakgd_recon'])
        
        err_pigdm = (gt - pigdm).abs()
        err_fakgd = (gt - fakgd).abs()
        
        vmax = gt.max().item()
        emax = max(err_pigdm.max().item(), err_fakgd.max().item())
        
        axes[i, 0].imshow(gt.numpy(), cmap='gray', vmin=0, vmax=vmax)
        axes[i, 0].set_title('Ground Truth' if i == 0 else '')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(pigdm.numpy(), cmap='gray', vmin=0, vmax=vmax)
        axes[i, 1].set_title(f'ΠGDM ({psnr_p:.2f} dB)' if i == 0 else f'{psnr_p:.2f} dB')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(fakgd.numpy(), cmap='gray', vmin=0, vmax=vmax)
        axes[i, 2].set_title(f'FA-KGD ({psnr_f:.2f} dB)' if i == 0 else f'{psnr_f:.2f} dB')
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(err_pigdm.numpy(), cmap='hot', vmin=0, vmax=emax)
        axes[i, 3].set_title('|Error| ΠGDM' if i == 0 else '')
        axes[i, 3].axis('off')
        
        im = axes[i, 4].imshow(err_fakgd.numpy(), cmap='hot', vmin=0, vmax=emax)
        axes[i, 4].set_title('|Error| FA-KGD' if i == 0 else '')
        axes[i, 4].axis('off')
    
    fig.suptitle(f'{title_prefix}', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

plot_comparison(slices_r4, n_show=3, title_prefix='EDM Model — R=4 Acceleration')

In [ ]:
plot_comparison(slices_r8, n_show=3, title_prefix='EDM Model — R=8 Acceleration')

## 3. PSNR trajectory comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (slices_data, meta, label) in zip(axes, [
    (slices_r4, meta_r4, 'R=4'),
    (slices_r8, meta_r8, 'R=8'),
]):
    for i, d in enumerate(slices_data):
        steps_p = np.arange(len(d['pigdm_psnr_traj']))
        steps_f = np.arange(len(d['fakgd_psnr_traj']))
        alpha = 0.4 if i > 0 else 1.0
        lbl_p = 'ΠGDM' if i == 0 else None
        lbl_f = 'FA-KGD+FPDC' if i == 0 else None
        ax.plot(steps_p, d['pigdm_psnr_traj'], 'b-', alpha=alpha, label=lbl_p)
        ax.plot(steps_f, d['fakgd_psnr_traj'], 'r-', alpha=alpha, label=lbl_f)
    
    ax.set_xlabel('Diffusion step')
    ax.set_ylabel('PSNR (dB)')
    ax.set_title(f'{label} — PSNR trajectories')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Frequency-domain error analysis

Decompose reconstruction error by k-space frequency band to see **where** FA-KGD improves.

In [ ]:
def freq_band_error(gt, recon, n_bands=5):
    """Compute NMSE per radial frequency band."""
    H, W = gt.shape
    radius_grid = build_radius_grid(H, W)
    r_max = radius_grid.max().item()
    band_edges = np.linspace(0, r_max, n_bands + 1)
    
    gt_k = fft2c(gt)
    recon_k = fft2c(recon)
    err_k = (gt_k - recon_k).abs() ** 2
    gt_k_power = gt_k.abs() ** 2
    
    nmse_bands = []
    for b in range(n_bands):
        ring = (radius_grid >= band_edges[b]) & (radius_grid < band_edges[b+1])
        if ring.sum() > 0:
            nmse_bands.append((err_k[ring].sum() / gt_k_power[ring].sum()).item())
        else:
            nmse_bands.append(0)
    return nmse_bands, band_edges

# Compute for all R=4 slices
n_bands = 5
pigdm_bands = np.zeros(n_bands)
fakgd_bands = np.zeros(n_bands)
n = len(slices_r4)

for d in slices_r4:
    pb, edges = freq_band_error(d['x_gt'], d['pigdm_recon'], n_bands)
    fb, _ = freq_band_error(d['x_gt'], d['fakgd_recon'], n_bands)
    pigdm_bands += np.array(pb)
    fakgd_bands += np.array(fb)

pigdm_bands /= n
fakgd_bands /= n

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(n_bands)
w = 0.35
labels = [f'{edges[b]:.0f}–{edges[b+1]:.0f}' for b in range(n_bands)]

bars1 = ax.bar(x_pos - w/2, pigdm_bands, w, label='ΠGDM', color='#3498db', alpha=0.8)
bars2 = ax.bar(x_pos + w/2, fakgd_bands, w, label='FA-KGD+FPDC', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Frequency band (radius)')
ax.set_ylabel('NMSE')
ax.set_title('Per-band k-space NMSE — R=4, EDM model')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Print improvement per band
for b in range(n_bands):
    if pigdm_bands[b] > 0:
        imp = (pigdm_bands[b] - fakgd_bands[b]) / pigdm_bands[b] * 100
        print(f'  Band {labels[b]:>10s}: ΠGDM={pigdm_bands[b]:.4f}, FA-KGD={fakgd_bands[b]:.4f}, improvement={imp:+.1f}%')

## 5. Ablation: Bug fixes that mattered

During development, we found two critical bugs:

1. **FPDC schedule direction**: The original implementation expanded frequencies from full→ACS (wrong). 
   Correct: ACS→full (low-freq first at high noise, expand to high-freq at low noise).

2. **M-step instability**: The online EM update explodes with real models because denoiser error 
   at high $\sigma_t$ dominates the innovation, inflating $\hat\sigma_i^2 \to \infty$ and 
   collapsing the Kalman gain $K \to 0$. Fix: clamp $\hat\sigma_i^2$ to never exceed initial estimate.

In [ ]:
# Load ablation results
ablation_runs = {
    'Broken FPDC + broken M-step': 'edm_test',
    'Broken FPDC + M-step off': 'edm_mstep_off',
    'Fixed FPDC + M-step off': 'edm_fpdc_fixed',
    'Fixed FPDC + M-step clamp': 'edm_fpdc_clamp',
    'Final (5 slices, R=4)': 'edm_5slices',
    'Final (5 slices, R=8)': 'edm_5slices_R8',
}

print(f'{"Configuration":<35s} {"ΠGDM":>8s} {"FA-KGD":>8s} {"Δ PSNR":>8s}  N')
print('-' * 70)

for label, run in ablation_runs.items():
    run_dir = OUTPUTS_DIR / run
    if not (run_dir / 'results.json').exists():
        continue
    with open(run_dir / 'results.json') as f:
        s = json.load(f).get('summary', {})
    print(f'{label:<35s} {s.get("pigdm_mean_psnr",0):>7.2f}  '
          f'{s.get("fakgd_mean_psnr",0):>7.2f}  '
          f'{s.get("delta_psnr_mean",0):>+7.2f}  '
          f'{s.get("num_slices","?"):>2}')

## 6. Summary results table

In [ ]:
# Final results summary
print('=' * 60)
print('FA-KGD+FPDC vs ΠGDM — EDM Score Network (65M params)')
print('=' * 60)
print(f'Model: EDMPrecond(SongUNet), trained on brain MRI (384×320)')
print(f'Data: fastMRI knee singlecoil test set (center-cropped to 384×320)')
print(f'Schedule: EDM (Karras), σ_min=0.002, σ_max=10, T=20 steps')
print(f'M-step mode: clamp (prevents denoiser-error inflation)')
print()

for run_name, label in [('edm_5slices', 'R=4'), ('edm_5slices_R8', 'R=8')]:
    with open(OUTPUTS_DIR / run_name / 'results.json') as f:
        d = json.load(f)
    s = d['summary']
    per_slice = d['per_slice']
    
    print(f'--- {label} ({s["num_slices"]} slices) ---')
    print(f'  ΠGDM:      {s["pigdm_mean_psnr"]:.2f} ± {np.std([r["pigdm_psnr"] for r in per_slice]):.2f} dB')
    print(f'  FA-KGD:    {s["fakgd_mean_psnr"]:.2f} ± {np.std([r["fakgd_psnr"] for r in per_slice]):.2f} dB')
    print(f'  Δ PSNR:    {s["delta_psnr_mean"]:+.2f} ± {s["delta_psnr_std"]:.2f} dB')
    print()

# Oracle comparison
print('--- Oracle denoiser (R=4, corrected FPDC, 3 slices) ---')
with open(OUTPUTS_DIR / 'oracle_fpdc_fixed' / 'results.json') as f:
    s = json.load(f)['summary']
print(f'  ΠGDM:      {s["pigdm_mean_psnr"]:.2f} dB')
print(f'  FA-KGD:    {s["fakgd_mean_psnr"]:.2f} dB')
print(f'  Δ PSNR:    {s["delta_psnr_mean"]:+.2f} dB')
print()
print('Note: The oracle denoiser isolates sampler quality from model quality.')
print('The +1.88 dB oracle gap vs +0.17 dB EDM gap suggests FA-KGD\'s advantage')
print('will grow with better score networks and more diffusion steps.')

## 7. Mask visualization

Show the undersampling mask used in the experiments.

In [ ]:
# Visualize the masks used
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, d_list, label in zip(axes, [slices_r4, slices_r8], ['R=4', 'R=8']):
    mask = d_list[0]['mask'].numpy()
    ax.imshow(mask, cmap='gray', aspect='auto')
    kept = mask.mean() * 100
    ax.set_title(f'{label} mask — {kept:.1f}% k-space lines acquired')
    ax.set_xlabel('k-space column')
    ax.set_ylabel('k-space row')

plt.tight_layout()
plt.show()

## Key takeaways

1. **FA-KGD+FPDC consistently outperforms ΠGDM** with a real score network: +0.17 dB at R=4, +0.15 dB at R=8 (tight std across slices).

2. **FPDC schedule direction matters**: Expanding from ACS→full (correct) vs full→ACS (wrong) is the difference between +0.17 dB and −0.37 dB.

3. **M-step requires stabilization**: The online EM update works well with oracle denoisers but is unstable with real models at high noise levels. Clamping $\hat\sigma_i^2$ to never exceed the initial ACS estimate is a simple and effective fix.

4. **Oracle gap (1.88 dB) ≫ EDM gap (0.17 dB)**: This indicates significant headroom — a better score network (or more steps, or matched training domain) would amplify FA-KGD's advantage.

5. **Domain mismatch**: The EDM model was trained on brain MRI (384×320) but evaluated on knee MRI. With matched data, we expect larger improvements.